# Work with Embeddings: Dimensionality Reduction and Clustering for Text Passages

This notebook takes text passages and vector embeddings and uses UMAP to project the vectors to 3D space for visualization. The complete structure (including original metadata and new coordinates) is saved as a JSON file.

## 0. Preliminaries

In [ ]:
# Get info about python version
import sys
print(sys.executable)
print(sys.version)
print(sys.version_info)

## 1. Setup

### 1.1 Install libraries

Instead of using the below python/ipython commands, and in order to make the notebook more declarative/reproducible, we try to define the necessary libraries and environment in a `uv` *project*, i.e. in the [pyproject.toml file](./pyproject.toml) that controls how `uv` manages the `.venv` virtual environment.

According to the [uv documentation](https://docs.astral.sh/uv/concepts/projects/layout/#the-project-environment):

> To run a command in the project environment, use `uv run`. Alternatively the project environment can be activated as normal for a virtual environment.
>
> When `uv run` is invoked, it will create the project environment if it does not exist yet or ensure it is up-to-date if it exists. The project environment can also be explicitly created with `uv sync`.
>
> It is *not* recommended to modify the project environment manually, e.g., with `uv pip install`. For project dependencies, use `uv add` to add a package to the environment.

### 1.2 Load libraries

In [ ]:
import pandas as pd
import numpy as np
import json
import logging
from umap import UMAP
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

### 1.3 Configuration

#### 1.3.1 File paths

In [ ]:
# Input files
file_path_in = './out-data'

# These files will be auto-detected from the most recent timestamp
# Priority order: parquet > pickle > csv (for docs)
# For embeddings: parquet > pickle

# We'll search for files with this pattern:
# YYYY-MM-DD_*_docs.{parquet,pkl,csv}
# YYYY-MM-DD_*_embeddings.{parquet,pkl}
# YYYY-MM-DD_*_processing_metadata.json
# YYYY-MM-DD_*_embedding_statistics.json

#### 1.3.2 Limits

In [ ]:
# Here we set the number of documents (paragraphs) to process
# Set it to a lower value until everything runs well, then increase it
# Set it to -1 to process all documents:
max_documents=-1

#### 1.3.3 Topic Modeling Configuration

In [ ]:
min_topic_size = 20   # how many documents must feature a topic for the topic to be legitimate
top_n_words = 35      # how many words to display in topic representation

## 2. Utility Functions

### 2.1 Logging Configuration

Configure structured logging for the embedding generation process.

In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

## 3. Read Input Data

### 3.1 Load document and embedding data

In [ ]:
def find_latest_files(directory: str, pattern: str) -> Optional[str]:
    """Find the most recent file matching the pattern."""
    files = glob.glob(os.path.join(directory, pattern))
    if not files:
        return None
    # Sort by modification time, most recent first
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

def load_docs_data(directory: str) -> pl.DataFrame:
    """Load docs data from parquet, pickle, or CSV, in that order."""
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_docs.parquet")
    if parquet_file:
        print(f"Loading docs from parquet: {parquet_file}")
        return pl.read_parquet(parquet_file)
    
    # Try pickle
    pickle_file = find_latest_files(directory, "*_docs.pkl")
    if pickle_file:
        print(f"Loading docs from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            df = pickle.load(f)
            # If it's a pandas DataFrame, convert to polars
            if hasattr(df, 'to_dict'):  # pandas DataFrame check
                return pl.from_pandas(df)
            return df
    
    # Try CSV
    csv_file = find_latest_files(directory, "*_docs.csv")
    if csv_file:
        print(f"Loading docs from CSV: {csv_file}")
        return pl.read_csv(csv_file)
    
    raise FileNotFoundError(f"No docs files found in {directory}")

def load_embeddings_data(directory: str) -> dict:
    """Load embeddings data from parquet or pickle, in that order.
    Returns a nested dict: {provider_name: {doc_id: [embedding_vector]}}
    """
    # Try parquet first
    parquet_file = find_latest_files(directory, "*_embeddings.parquet")
    if parquet_file:
        print(f"Loading embeddings from parquet: {parquet_file}")
        # The parquet was created from a nested dict: {provider: {doc_id: embedding}}
        # When saved with pl.DataFrame(nested_dict).write_parquet(), 
        # it creates columns named after providers, with each cell containing the doc_id->embedding mapping
        df_embeddings = pl.read_parquet(parquet_file)
        
        # Convert back to nested dict structure
        # Each column represents a provider, column values are the {doc_id: embedding} dicts
        embeddings_dict = {}
        for column_name in df_embeddings.columns:
            # Get the provider's embeddings (should be a single row with a dict/struct)
            provider_data = df_embeddings[column_name][0]
            if provider_data is not None:
                embeddings_dict[column_name] = provider_data
        
        return embeddings_dict
    
    # Try pickle
    pickle_file = find_latest_files(directory, "*_embeddings.pkl")
    if pickle_file:
        print(f"Loading embeddings from pickle: {pickle_file}")
        with open(pickle_file, "rb") as f:
            return pickle.load(f)
    
    raise FileNotFoundError(f"No embeddings files found in {directory}")

def load_metadata(directory: str) -> dict:
    """Load processing metadata JSON."""
    metadata_file = find_latest_files(directory, "*_processing_metadata.json")
    if metadata_file:
        print(f"Loading metadata from: {metadata_file}")
        with open(metadata_file, "r") as f:
            return json.load(f)
    return {}

# Load all data
print("="*80)
print("Loading data files...")
print("="*80)

docs = load_docs_data(file_path_in)
embeddings_dict = load_embeddings_data(file_path_in)
metadata = load_metadata(directory=file_path_in)

print(f"\nLoaded {docs.height} documents")
print(f"Loaded embeddings for {len(embeddings_dict)} providers:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  - {provider}: {len(provider_embeddings)} documents")

if metadata:
    print(f"\nProcessing metadata:")
    print(f"  Processing date: {metadata.get('processing_date', 'N/A')}")
    if 'providers' in metadata:
        print(f"  Providers in metadata: {list(metadata['providers'].keys())}")

#### 3.1.1 Data Structure Overview

The loaded data has the following structure:
- **docs**: Polars DataFrame with document metadata (url, content, xmlid, author, etc.)
- **embeddings_dict**: Nested dictionary organized as:
  ```python
  {
      "provider_name_1": {
          "doc_id_1": [embedding_vector_1],
          "doc_id_2": [embedding_vector_2],
          ...
      },
      "provider_name_2": {
          "doc_id_1": [embedding_vector_1],
          ...
      }
  }
  ```
- **metadata**: Configuration and processing information from the embedding creation run

Now, give some information about the data:

In [ ]:
print("="*80)
print("Data Overview")
print("="*80)
print(f"\nShape of docs dataframe: {docs.shape}")
print(f"Number of available documents: {docs.height}")
print(f"\nDocument length statistics:")
if "len_content" in docs.columns:
    print(docs["len_content"].describe())
else:
    print("  (len_content column not available)")

print("\nEmbeddings by provider:")
for provider, provider_embeddings in embeddings_dict.items():
    print(f"  {provider}: {len(provider_embeddings)} embeddings")
    # Show embedding dimension
    if provider_embeddings:
        sample_embedding = next(iter(provider_embeddings.values()))
        print(f"    - Dimension: {len(sample_embedding)}")

print("\nFirst 3 rows of docs dataframe:")
docs.head(3)

### 3.2 Parse the Embeddings Column

In [ ]:
# Convert the list of embeddings to a numpy array for UMAP
embeddings_array = np.array(df['embedding_vectors'].tolist())
print(f"Embeddings shape: {embeddings_array.shape}")

## 4. Run UMAP to Project Vectors to 3D Space

In [ ]:
# Configure UMAP
print("Running UMAP dimensionality reduction...")
umap_3d = UMAP(n_components=3,  # Project to 3D
               n_neighbors=15,   # Size of local neighborhood (adjust based on your dataset)
               min_dist=0.1,     # Minimum distance between points in the projection
               metric='cosine',  # Typically used for embeddings
               random_state=42)  # For reproducibility

# Fit and transform the data
embeddings_3d = umap_3d.fit_transform(embeddings_array)

# Add the 3D coordinates to the dataframe
df['umap_x'] = embeddings_3d[:, 0]
df['umap_y'] = embeddings_3d[:, 1]
df['umap_z'] = embeddings_3d[:, 2]

print(f"UMAP projection complete. Shape: {embeddings_3d.shape}")

## 5. Visualize the 3D Projection

In [ ]:
# Create a 3D scatter plot to visualize the projection
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# If there's a topic column, we can color by topic
if 'topic' in df.columns and df['topic'].nunique() < 20:  # Only if there are a reasonable number of topics
    # Convert topics to numeric if they're not already
    if df['topic'].dtype == 'object':
        df['topic_numeric'] = pd.Categorical(df['topic']).codes
    else:
        df['topic_numeric'] = df['topic']
    
    scatter = ax.scatter(
        df['umap_x'], 
        df['umap_y'], 
        df['umap_z'],
        c=df['topic_numeric'], 
        cmap='viridis', 
        alpha=0.7,
        s=10
    )
    
    # Add a colorbar
    cbar = plt.colorbar(scatter)
    cbar.set_label('Topic')
else:
    # Simple scatter plot without coloring by topic
    ax.scatter(
        df['umap_x'], 
        df['umap_y'], 
        df['umap_z'],
        alpha=0.7,
        s=10
    )

ax.set_title('3D UMAP Projection of Text Embeddings')
ax.set_xlabel('UMAP Dimension 1')
ax.set_ylabel('UMAP Dimension 2')
ax.set_zlabel('UMAP Dimension 3')

plt.tight_layout()
plt.show()

## 6. Save the Complete Structure as JSON

In [ ]:
# Option to include or exclude embeddings in the output file
include_embeddings = False  # Set this to True if you want to include embeddings in the output

# Remove the new embedding_vectors column (we already have the original embeddings column)
if 'embedding_vectors' in df.columns:
    df = df.drop(columns=['embedding_vectors'])

# If embeddings should be excluded, remove the embeddings column
if not include_embeddings and 'embeddings' in df.columns:
    print("Excluding embeddings from output JSON to reduce file size")
    df = df.drop(columns=['embeddings'])

for c in ['topic',
          'topic-label',
          'topic-label2',
          'topic-description',
          'lang',
          'author-id',
          'content',
          'len_content']:
    if c in df.columns:
        df = df.drop(columns=c)

print(df.columns.tolist())

In [ ]:

# Convert DataFrame to JSON
print("Converting to JSON...")
json_data = df.to_json(orient='records')

# Save the JSON to a file
# Create output filename based on whether embeddings are included
if include_embeddings:
    output_json_file = 'data/passages_with_umap_3d_with_embeddings.json'
else:
    output_json_file = 'data/passages_with_umap_3d.json'

with open(output_json_file, 'w') as f:
    f.write(json_data)

print(f"Saved JSON to {output_json_file}")

# Show the file size
import os
file_size_mb = os.path.getsize(output_json_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

## 7. Data Exploration and Analysis of the 3D Projection

In [ ]:
# Let's examine the distribution of points in the UMAP space
print("UMAP 3D Coordinates Statistics:")
print(df[['umap_x', 'umap_y', 'umap_z']].describe())

# Check for any clusters or patterns if topic labels are present
if 'topic' in df.columns and df['topic'].nunique() < 20:
    print("\nPoint distribution by topic:")
    topic_counts = df['topic'].value_counts()
    print(topic_counts)
    
    # Calculate cluster centroids by topic
    print("\nCluster centroids by topic:")
    centroids = df.groupby('topic')[['umap_x', 'umap_y', 'umap_z']].mean()
    print(centroids)

In [ ]:
# Create a sample of the JSON output to examine the structure
sample_json = json.loads(df.iloc[:2].to_json(orient='records'))
print("Sample of JSON output (first 2 records):")
print(json.dumps(sample_json, indent=2)[:1000] + "...")

## 8. Conclusion

This notebook has successfully:
1. Loaded the CSV file with text passages and embeddings
2. Parsed the vector embeddings from the string format
3. Applied UMAP to project the high-dimensional vectors to 3D space
4. Visualized the 3D projection
5. Saved the complete structure (all original columns plus the 3D coordinates) as a JSON file

The JSON file can now be used for further analysis or visualization in tools like Plotly, D3.js, or other data visualization libraries.